### Multi-Modal Input Exploration (multiple files incl. text pdfs, diagrams, etc)

In [ ]:
import os
from PIL import Image
import google.generativeai as genai
from llama_index.core import Document, VectorStoreIndex, Settings, SimpleDirectoryReader
from llama_index.llms.gemini import Gemini
import utils

Set up LLM and embedding model (turning text into vectors for retrieval)

In [9]:
Settings.llm = Gemini(model="models/gemini-2.5-flash", api_key=utils.get_gemini_api_key(), temperature=0.1)
Settings.embed_model = "local:BAAI/bge-small-en-v1.5"

C:\Users\Toby\AppData\Local\Temp\ipykernel_2028\1006625785.py:1: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/This package will no longer be supported after version 0.6.2) -- Deprecated since version 0.6.2.
  Settings.llm = Gemini(model="models/gemini-2.5-flash", api_key=utils.get_gemini_api_key(), temperature=0.1)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7363.36it/s]


Setting up vision capable model using google's genai package

In [10]:
genai.configure(api_key=utils.get_gemini_api_key())
vision_model = genai.GenerativeModel("gemini-2.5-flash")

Retry helper for handling rate limits

- adds a timer between API calls if rate limit is exceeded, prevents crashes
- more likely to hit rate limit on image-based calls since they're heavier than text-only

In [ ]:
def call_with_retry(model, parts, max_retries=5, base_delay=10):
    for attempt in range(max_retries):
        try:
            return model.generate_content(parts)
        except ResourceExhausted:
            if attempt == max_retries - 1:
                raise
            wait = base_delay * (2 ** attempt)
            print(f"Rate limited, retrying in {wait}s...")
            time.sleep(wait)

Read through text documents and turn them into Document objects

- Llama-index functions like SentenceSplitter and SemanticSplitter expect Document objects as input
- can attach metadata to Document objects (page number, file name, type of object "text")

In [11]:
# 1. text documents (PDFs)
text_docs = SimpleDirectoryReader(
    input_files=[
        "knowledge_base/backprop_old.pdf",
        "knowledge_base/synergizing_reasonsing_llm.pdf",
    ]
).load_data()
for d in text_docs:
    d.metadata["type"] = "text"

In [13]:

# 2. caption each image into a Document with the same "shape" as text docs
image_paths = [
    "knowledge_base/diagram1.png",
    "knowledge_base/otters.png",
    "knowledge_base/poster.png",
    "knowledge_base/resume.png",
]

import json

CAPTION_CACHE = "knowledge_base/.captions.json"

def load_or_caption(image_paths, vision_model, cache_path=CAPTION_CACHE):
    cache = {}
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            cache = json.load(f)

    for path in image_paths:
        if path not in cache:
            img = Image.open(path)
            cache[path] = vision_model.generate_content(
                ["Describe this image in detail, including any text, diagrams, or data visible.", img]
            ).text
            with open(cache_path, "w") as f:
                json.dump(cache, f)  # save after each call so partial progress survives a crash

    return cache

captions = load_or_caption(image_paths, vision_model)
image_docs = [
    Document(text=captions[p], metadata={"file_path": p, "type": "image", "file_name": os.path.basename(p)})
    for p in image_paths
]


In [14]:
# --- 3. ONE combined index -> ONE retrieval step returns a mix of text + image nodes ---
index = VectorStoreIndex.from_documents(text_docs + image_docs)
query_engine = index.as_query_engine(similarity_top_k=5)

In [15]:
# --- 4. query function: retrieves text + image nodes, grounds final answer in real images ---
def multimodal_query(question: str) -> str:
    response = query_engine.query(question)

    retrieved_images = [
        n.metadata["file_path"]
        for n in response.source_nodes
        if n.metadata.get("type") == "image"
    ]

    if not retrieved_images:
        return str(response)  # only text was relevant, no grounding needed

    parts = [question, "\n\nRetrieved context:\n" + str(response)]
    parts += [Image.open(p) for p in retrieved_images]

    return call_with_retry(vision_model, parts).text

In [18]:
print(multimodal_query("What's shown in the architecture diagram, and how does it relate to the ReAct paper?"))


The architecture diagram illustrates a comprehensive AI agent system, broken down into two main operational areas: "Harness & Loop" and "LLM Ops."

**What's shown in the architecture diagram:**

1.  **Harness & Loop:** This section describes the core functionality and memory management of the AI agent:
    *   **Input and Working Memory:** User Prompts, Current Chat History, and System Prompts are fed into a "Working Memory / Context RAM."
    *   **Retrieval Augmented Generation (RAG):** The working memory accesses external knowledge:
        *   **Procedural Memory:** Stores "how-to-act" skills and instructions (e.g., "Skill.md").
        *   **Semantic Memory:** Stores "durable facts" and "user profiles" (using a Vector store, with RAG for relevance and SQL for recency).
    *   **LLM - Q&A Agent:** A large language model (e.g., GPT, Claude) processes the combined context.
    *   **Agentic Tools:** The LLM can make "Tool Calls" to external agentic tools (e.g., schedule meetings, re